# L12c: Deep Q-Learning
In this lecture, we replace the Q-table with a function that computes the action values from a few numbers that describe the state. This idea is the basis of deep Q-learning (DQN). We build it with the simplest such function, a linear one.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Explain why a Q-table runs out of room:__ A table needs one entry for every state and action, and the number of states grows exponentially with the number of state variables. We'll also see that a table learns nothing about a state from the states around it.
> * __Write the DQN loss and its gradient step:__ DQN fits a Q-network to bootstrap targets by gradient descent on a squared error. We'll derive the gradient step for a linear Q-network and show that tabular Q-learning is a special case of it.
> * __Describe the replay buffer and the target network:__ The replay buffer trains the network on random samples of past experience, and the target network holds the targets fixed between copies. We'll see which problem of Q-learning with a network each one addresses.

Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Learn a dosing policy without a grid using a linear DQN](CHEME-5800-L12c-Example-LinearDQN-AdaptiveDosing-Fall-2026.ipynb). In this example, an agent learns to dose the patient of the L12b lab. The tumor burden and the toxicity stay continuous numbers instead of being rounded to grid levels.
>
> The agent's linear Q-network has 148 weights, where the L12b Q-table has 6,724 entries. We compare its doses with a policy that value iteration computes on a much finer grid.

___

## The Curse of Dimensionality
Tabular Q-learning stores one number for every state-action pair. The table runs out of room in three ways:
* __The table grows exponentially.__ A state with $m$ variables, each rounded to $n$ levels, has $n^{m}$ states. The L12b patient has two variables on 41 levels, which gives $41^{2}=1681$ states. Ten variables on 41 levels would give about $1.3\times 10^{16}$ states, far more than an agent could ever visit.
* __The table does not generalize.__ Each entry learns only from visits to its own state-action pair. Two states with almost the same tumor burden share nothing. A state the agent never visits keeps its starting value.
* __Rounding changes the problem.__ A continuous state, such as a tumor burden, must be rounded to a grid level. A coarse grid treats different states as the same state. In the example, the L12b policy gives no dose to a patient with a tumor burden of 0.109, because the grid rounds 0.109 to 0.100, a level that counts as a cure.

Richard Bellman called this exponential growth the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality). The way around it is to replace the table with a function that computes $Q(s,a)$ from a few numbers that describe the state. What the agent learns in one state then carries over to the states around it.

___

## Q-Networks
A __Q-network__ is a function $Q(s,a;\theta)$ with adjustable parameters $\theta$ that returns the value of action $a$ in state $s$. It takes the state as its input and returns a vector of Q-values, one per action. The greedy policy picks the action with the largest entry, $\pi(s)=\arg\max_{a\in\mathcal{A}}Q(s,a;\theta)$.

Learning now means adjusting $\theta$ instead of the entries of a table. The number of parameters is set by the network, not by the number of states.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Q-Networks/Fig-Q-Networks.svg" width="720" style="max-width:100%; height:auto;" alt="Two diagrams, each mapping a state s to one Q-value per action, Q of s and a 1 through Q of s and a n. The top diagram, labeled linear Q-network, sends the state through gray arrows to a column of feature circles phi 1, phi 2, down to phi p, labeled features phi of s, fixed. Blue lines, labeled weights W, learned, connect every feature to every Q-value. The bottom diagram, labeled deep Q-network, sends the state into a shaded box of three hidden layers of circles joined by blue lines, labeled hidden layers, parameters theta, learned. One arrow leaves the box and fans out to the Q-values."/>
</p>

__Two Q-networks.__ Both networks map a state $s$ to one Q-value per action. In the linear network, only the weights (blue) are learned, and the features are fixed functions of the state. In the deep network, the hidden layers are learned as well.

### A linear Q-network
The simplest Q-network is linear. It describes the state $s$ by a __feature vector__ $\phi(s)\in\mathbb{R}^{p}$, a list of $p$ numbers computed from the state. Each action $a$ has a weight vector $\mathbf{w}_{a}\in\mathbb{R}^{p}$, and its Q-value is the weighted sum of the features:

$$
\begin{equation*}
Q(s,a;\mathbf{W}) = \mathbf{w}_{a}^{\top}\phi(s) = \sum_{j=1}^{p}w_{a,j}\,\phi_{j}(s)
\end{equation*}
$$

where the weight matrix $\mathbf{W}$ has one row $\mathbf{w}_{a}^{\top}$ per action. The network has $|\mathcal{A}|\cdot p$ weights.

__The features.__ In the example, the state is the pair $s=(B,Z)$ of tumor burden and toxicity. Its features are __radial basis functions__, smooth bumps with centers $\mathbf{c}_{j}$ spread over the square of states:

$$
\begin{equation*}
\phi_{j}(s)=\exp\left(-\frac{\lVert s-\mathbf{c}_{j}\rVert^{2}}{2\sigma^{2}}\right)
\end{equation*}
$$

where $\lVert s-\mathbf{c}_{j}\rVert$ is the distance from the state to center $j$, and $\sigma$ sets the width of every bump. Bump $j$ equals one at its center and falls toward zero away from it.

Nearby states have similar features. Their Q-values are then similar too. This is how what the agent learns in one state carries over to its neighbors.

__A Q-table is a linear network.__ Number the states $1,2,\dots,|\mathcal{S}|$, and let $\phi(s)$ be the __one-hot__ vector, with a one in entry $s$ and zeros everywhere else. Then $Q(s,a;\mathbf{W})=w_{a,s}$, and the weight matrix is the Q-table.

### A deep Q-network
A __deep Q-network__ replaces the fixed features with the hidden layers of a __neural network__. A neural network is a function built from layers. Each layer takes weighted sums of its inputs and passes each sum through a fixed nonlinear function, such as $\max(0,x)$.

The hidden layers learn their own features from the data. [Mnih et al. (2015)](https://www.nature.com/articles/nature14236) trained a deep Q-network to play Atari video games from the pixels on the screen.

The rest of this lecture holds for any Q-network. We write $\theta$ for its parameters, which are the weights $\mathbf{W}$ in the linear case.

___

## The DQN Loss
Q-learning moves the entry $Q(s,a)$ toward the target $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$. A Q-network has no entry to move. Instead, DQN adjusts $\theta$ to shrink the squared gap between the network's prediction and the target, as in a regression.

The training data are __transitions__ $(s_{i},a_{i},r_{i},s_{i}^{\prime},d_{i})$. Each one holds a state, the action taken, the reward, the next state, and a __done flag__ $d_{i}$, which is one if the episode ended at $s_{i}^{\prime}$ and zero otherwise.

> __DQN Bellman Loss:__
>
> The target of transition $i$ is
>
> $$y_{i} = r_{i} + \gamma\,(1-d_{i})\max_{a^{\prime}\in\mathcal{A}}Q(s_{i}^{\prime},a^{\prime};\theta^{-})$$
>
> where $\theta^{-}$ are the parameters of the __target network__, a delayed copy of the Q-network described below. The factor $(1-d_{i})$ drops the future value when the episode has ended.
>
> The Q-network is trained on a __mini-batch__ of $B$ transitions by gradient descent on the mean squared error:
>
> $$\mathcal{L}(\theta) = \frac{1}{B}\sum_{i=1}^{B}\left(y_{i}-Q(s_{i},a_{i};\theta)\right)^{2}$$
>
> The prediction uses the action $a_{i}$ that was actually taken. The target uses the best next action under the target network.

The loss compares one entry of the network's output, the one for the action taken, with its target. The other actions have no target in this transition.

### The gradient step of a linear network
For the linear network, the prediction is $Q(s_{i},a_{i};\mathbf{W})=\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})$. Its derivative with respect to $\mathbf{w}_{a_{i}}$ is the feature vector $\phi(s_{i})$, and the other rows of $\mathbf{W}$ do not appear in it. The target $y_{i}$ does not depend on $\mathbf{W}$, because the target network computes it.

One gradient step with learning rate $\alpha$ then updates the row of each action in the mini-batch:

$$
\begin{equation*}
\mathbf{w}_{a}\gets\mathbf{w}_{a}+\frac{\alpha}{B}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i})
\end{equation*}
$$

where the sum runs over the transitions of the mini-batch that took action $a$, and the factor 2 from the square is folded into $\alpha$. Each transition adds its prediction error times its features. The logistic regression step of week 9 has the same form.

__A check against the table.__ Take one-hot features, a mini-batch of one transition, and a target network that is always equal to the Q-network. The update then reads $Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$. This is the Q-learning update of L12a.

___

## Two Stabilizers
Tabular Q-learning converges under the conditions we saw in L12a. Q-learning with a network has no such guarantee. Its values can even grow without bound, a failure called __divergence__ ([Sutton and Barto, 2018, Chapter 11](http://incompleteideas.net/book/the-book-2nd.html)).

DQN adds two stabilizers that make training much steadier in practice: a replay buffer and a target network.

### The replay buffer
Consecutive transitions of one episode are alike. In the example, the patient's state changes only a little from one cycle to the next. A run of gradient steps on such transitions pulls the network toward one region of states and spoils its values elsewhere.

DQN stores the most recent $M$ transitions in a __replay buffer__ $\mathcal{D}$. Each training step uses a mini-batch of $B$ transitions drawn at random from the buffer. A random mini-batch mixes transitions from many episodes, and each transition is reused in many steps.

Training waits until the buffer holds $N_{\text{warm}}$ transitions, a period called the __warm-up__. Without it, the first mini-batches would draw again and again from the few transitions of the first episode.

### The target network
The target $y_{i}$ depends on the network itself. If one set of parameters computed both the prediction and the target, each step would also move the target. The network would then chase a target that moves with it.

DQN computes the targets with the __target network__ $Q(s,a;\theta^{-})$, a copy of the Q-network. Every $C$ steps, we copy the Q-network into it, $\theta^{-}\gets\theta$. Between copies, the targets stay fixed, and each stretch of $C$ steps is a regression with fixed labels.

___

## The DQN Algorithm
DQN runs in episodes, like Q-learning. Each step acts epsilon-greedily from the Q-network, stores the transition, and, after the warm-up, takes one gradient step on a random mini-batch.

__Initialize__: Given a Q-network $Q(s,a;\theta)$, a discount factor $0\leq\gamma<1$, a learning rate $\alpha>0$, an exploration probability $\epsilon_{i}$ for each episode $i$, a buffer size $M$, a warm-up size $N_{\text{warm}}$, a mini-batch size $B$, a copy interval $C$, a number of episodes $N$, and a maximum number of steps per episode $t_{\max}$, set the parameters $\theta$ (to zeros or small random values), copy them to the target network, $\theta^{-}\gets\theta$, and start with an empty replay buffer $\mathcal{D}$.

For $i = 1$ to $N$ __do__:
1. Choose a starting state $s$ that is not absorbing, and set the step counter $t\gets 0$.
2. While $s$ is not absorbing and $t<t_{\max}$ __do__:
    - Choose the action $a$: roll a random number $p\in[0,1]$.
        - If $p\leq\epsilon_{i}$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime};\theta)$ (exploitation).
    - Take action $a$, and observe the reward $r$, the next state $s^{\prime}$, and the done flag $d$ from the world.
    - Store the transition $(s,a,r,s^{\prime},d)$ in $\mathcal{D}$. If $\mathcal{D}$ now holds more than $M$ transitions, remove the oldest one.
    - If $\mathcal{D}$ holds at least $N_{\text{warm}}$ transitions, take one training step:
        - Draw a mini-batch of $B$ transitions $(s_{j},a_{j},r_{j},s_{j}^{\prime},d_{j})$ at random from $\mathcal{D}$.
        - Compute each target $y_{j}\gets r_{j}+\gamma(1-d_{j})\max_{a^{\prime}\in\mathcal{A}}Q(s_{j}^{\prime},a^{\prime};\theta^{-})$.
        - Take one gradient step on the loss: $\theta\gets\theta-\alpha\nabla_{\theta}\mathcal{L}(\theta)$.
    - Every $C$ steps, copy the Q-network to the target network: $\theta^{-}\gets\theta$.
    - Update the state $s\gets s^{\prime}$ and the step counter $t\gets t+1$.

__Extract Policy__: For any state $s$, the policy is $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a;\theta)$.

The policy works for any state, including states the agent never visited. A Q-table can only give a policy for the states in its rows.

__Why only one gradient step per mini-batch?__ A mini-batch is a small random sample of the buffer. Many steps on one mini-batch would fit those few transitions and undo what the network learned from the rest.

In the example, a linear network with 37 features per action trains for 3,000 episodes in about one second. The same algorithm trains a deep network, which needs a package for neural networks and much more time.

___

## Overestimation
The target takes the maximum over the estimated values of the next state. When those estimates are noisy, their maximum is too large on average, even if each estimate is right on average. The update passes this upward bias back to earlier states.

__Double DQN__ ([van Hasselt, Guez, and Silver, 2016](https://arxiv.org/abs/1509.06461)) changes one thing in the target. The Q-network picks the best next action, and the target network scores it:

$$
\begin{equation*}
y_{i} = r_{i} + \gamma\,(1-d_{i})\,Q\left(s_{i}^{\prime},\;\arg\max_{a^{\prime}\in\mathcal{A}}Q(s_{i}^{\prime},a^{\prime};\theta);\;\theta^{-}\right)
\end{equation*}
$$

Noise that makes one action look best to the Q-network is less likely to inflate that action's score in the target network. In the lab, we measure the bias and test this fix.

___

## Lab
In the [lab](../L12d/CHEME-5800-L12d-Lab-DoubleDQN-Fall-2026.ipynb), we see where overestimation comes from on a small problem with two states. We compare Q-learning with Double Q-learning, then DQN with Double DQN.

___

## Summary
In this lecture, we replaced the Q-table with a Q-network and built deep Q-learning around it.

> __Key Takeaways:__
>
> * __A Q-network replaces the table:__ A table needs one entry per state and action and learns nothing about a state from its neighbors. A Q-network computes every action value from features of the state, with a number of parameters that does not grow with the number of states.
>
> * __DQN trains the network by regression on bootstrap targets:__ Each target is the reward plus the discounted best value of the next state under the target network. For a linear network, one gradient step adds each prediction error times the features to the weights of the action taken.
>
> * __The replay buffer and the target network stabilize training:__ Random mini-batches from the replay buffer mix transitions from many episodes and reuse each one many times. The target network holds the targets fixed between copies, which keeps the network from chasing a target that moves with it.

A linear network with well-chosen features is often enough for a problem with a few state variables. A deep network learns its own features, which lets the same algorithm work from raw inputs such as images.

___